In [1]:
# Import Libraries

import gc
import sys
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import matplotlib.pyplot as plt

import golois

print ("Python version", sys.version_info)
print ("Tensorflow version", tf.__version__)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def plot_learning_rate(lrs):
    plt.figure(figsize=(10, 5))
    plt.plot(lrs)
    plt.title('Learning Rate Evolution During Training')
    plt.xlabel('Batch')
    plt.ylabel('Learning Rate')
    plt.grid(True)
    plt.show()
    
def print_validation_results(model_results, epoch=100):
    for model, val, label, time in model_results:
        metrics = dict(zip(model.metrics_names, val))
        title = f"📊 Validation Results for {label}"
        if epoch is not None:
            title += f" — Epoch {epoch}"
        print(f"\n{title}:")
        for name, value in metrics.items():
            print(f"  - {name:<30}: {value:.4f}")
        print(f"  - Time: {time:.4f}")

def plot_result(history_dfs, val_dfs, labels, epochs=None):
    assert len(history_dfs) == len(labels)

    title = f"Epochs: {epochs}"

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3)

    # --- Ligne 1 : 3 plots ---
    ax1 = fig.add_subplot(gs[0, 0])
    for df, val_df, label in zip(history_dfs, val_dfs, labels):
        ax1.plot(df['epoch'], df['loss'], label=f'{label} Train Loss')
        if val_df is not None:
            if 'val_policy_loss' in val_df.columns and 'val_value_loss' in val_df.columns:
                val_total_loss = val_df['val_policy_loss'] + val_df['val_value_loss']
                ax1.plot(val_df['epoch'], val_total_loss, 'o--', label=f'{label} Val Loss (recalculated)')
    ax1.set_title('Total Loss par Epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Total Loss')
    ax1.legend()

    ax2 = fig.add_subplot(gs[0, 1])
    for df, val_df, label in zip(history_dfs, val_dfs, labels):
        if 'policy_loss' in df.columns:
            ax2.plot(df['epoch'], df['policy_loss'], label=f'{label} Train Policy Loss')
        if val_df is not None and 'val_policy_loss' in val_df.columns:
            ax2.plot(val_df['epoch'], val_df['val_policy_loss'], 'o--', label=f'{label} Val Policy Loss')
    ax2.set_title('Policy Loss par Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Policy Loss')
    ax2.legend()

    ax3 = fig.add_subplot(gs[0, 2])
    for df, val_df, label in zip(history_dfs, val_dfs, labels):
        if 'value_loss' in df.columns:
            ax3.plot(df['epoch'], df['value_loss'], label=f'{label} Train Value Loss')
        if val_df is not None and 'val_value_loss' in val_df.columns:
            ax3.plot(val_df['epoch'], val_df['val_value_loss'], 'o--', label=f'{label} Val Value Loss')
    ax3.set_title('Value Loss par Epoch')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Value Loss')
    ax3.legend()

    # --- Ligne 2 : 2 plots ---
    ax4 = fig.add_subplot(gs[1, 0])
    for df, label in zip(history_dfs, labels):
        if 'policy_categorical_accuracy' in df.columns:
            ax4.plot(df['epoch'], df['policy_categorical_accuracy'], label=f'{label} Train Policy Acc')
    ax4.set_title('Policy Accuracy par Epoch')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Categorical Accuracy')
    ax4.legend()

    ax5 = fig.add_subplot(gs[1, 1])
    for df, label in zip(history_dfs, labels):
        if 'value_mse' in df.columns:
            ax5.plot(df['epoch'], df['value_mse'], label=f'{label} Train Value MSE')
    ax5.set_title('Value MSE par Epoch')
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('MSE')
    ax5.legend()

    # Libérer la dernière case vide
    fig.delaxes(fig.add_subplot(gs[1, 2]))

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


Python version sys.version_info(major=3, minor=9, micro=21, releaselevel='final', serial=0)
Tensorflow version 2.15.0


In [12]:
# v2.0.2.1 Add kernel_initializer='he_normal'

from tensorflow import keras
from keras import regularizers
from keras.models import Model
from keras.layers import (
    Input, Dense, Conv2D, GlobalAveragePooling2D, Dropout, Flatten,
    Activation, BatchNormalization, Add, Reshape, DepthwiseConv2D, Multiply
)
from keras.utils import plot_model

def _se_block(input_tensor, filters, ratio=16, activation=keras.activations.swish):
    se = GlobalAveragePooling2D()(input_tensor)
    se = Reshape((1, 1, filters))(se)
    se = Dense(filters // ratio, activation=activation, use_bias=False,
               kernel_initializer='he_normal')(se)
    se = Dense(filters, activation='sigmoid', use_bias=False,
               kernel_initializer='he_normal')(se)
    return Multiply()([input_tensor, se])

def _conv_block(inputs, filters, kernel, activation=keras.activations.swish):
    x = Conv2D(filters, kernel, padding='same', kernel_regularizer=regularizers.l2(0.0001),
               use_bias=False, kernel_initializer='he_normal')(inputs)
    x = BatchNormalization(axis=1)(x)
    x = Activation(activation)(x)
    return x

def _bottleneck_block(inputs, filters, kernel, factor, se, activation=keras.activations.swish):
    expanded_filters = filters * factor

    x = _conv_block(inputs, filters=expanded_filters, kernel=(1, 1), activation=activation)
    x = DepthwiseConv2D(kernel, padding='same', kernel_regularizer=regularizers.l2(0.0001),
                        use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization(axis=1)(x)
    x = _conv_block(x, filters=filters, kernel=(1, 1), activation=activation)

    if se:
        x = _se_block(x, filters, 16, activation)

    x = Add()([x, inputs])
    x = Activation(activation)(x)
    return x

def GoMobileNetv2(input_shape, filters, factor, block_num, se, activation=keras.activations.swish, drop_out_rate=0.3):
    inputs = Input(shape=input_shape)
    x = _conv_block(inputs, filters, (1, 1), activation=activation)

    for i in range(block_num):
        x = _bottleneck_block(x, filters, (3, 3), factor, se, activation=activation)

    policy_head = _conv_block(x, filters=1, kernel=(1, 1), activation=activation)
    policy_head = Flatten()(policy_head)
    policy_head = Activation('softmax', name='policy')(policy_head)

    value_head = GlobalAveragePooling2D()(x)
    value_head = Dense(50, kernel_regularizer=regularizers.l2(0.0001),
                       kernel_initializer='he_normal')(value_head)
    value_head = Activation(activation)(value_head)
    value_head = Dropout(drop_out_rate)(value_head)
    value_head = Dense(1, activation='sigmoid', name='value',
                       kernel_regularizer=regularizers.l2(0.0001),
                       kernel_initializer='he_normal')(value_head)

    model = keras.Model(inputs=inputs, outputs=[policy_head, value_head])
    return model


In [7]:
# v2.0.2.1 Set SGD clipnorm=1.0

import time
import pandas as pd
import gc
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import optimizers, backend as K
from tensorflow.keras.callbacks import Callback

class LrLogger(Callback):
    def __init__(self):
        super().__init__()
        self.lrs = []

    def on_train_batch_end(self, batch, logs=None):
        lr = float(K.get_value(self.model.optimizer.lr))
        self.lrs.append(lr)
class CyclicLR(Callback):
    def __init__(self, base_lr=1e-4, max_lr=1e-2, step_size=2000, mode='triangular'):
        super(CyclicLR, self).__init__()
        self.base_lr = base_lr
        self.max_lr = max_lr
        self.step_size = step_size
        self.mode = mode
        self.iterations = 0.
        self.history = {}

    def clr(self):
        cycle = np.floor(1 + self.iterations / (2 * self.step_size))
        x = np.abs(self.iterations / self.step_size - 2 * cycle + 1)
        if self.mode == 'triangular':
            return self.base_lr + (self.max_lr - self.base_lr) * max(0, (1 - x))
        else:
            raise ValueError('Only "triangular" mode is implemented.')

    def on_train_batch_begin(self, batch, logs=None):
        self.iterations += 1
        lr = self.clr()
        K.set_value(self.model.optimizer.lr, lr)
        self.history.setdefault('lr', []).append(lr)

def train_model(model, batch=32, policy_weight=1.0, value_weight=1.0, epochs=100, N=10000):
    start_time = time.time()
    
    # Configuration
    planes = 31
    moves = 361

    input_data = np.random.randint(2, size=(N, 19, 19, planes))
    input_data = input_data.astype ('float32')

    policy = np.random.randint(moves, size=(N,))
    policy = keras.utils.to_categorical (policy)

    value = np.random.randint(2, size=(N,))
    value = value.astype ('float32')

    end = np.random.randint(2, size=(N, 19, 19, 2))
    end = end.astype ('float32')

    groups = np.zeros((N, 19, 19, 1))
    groups = groups.astype ('float32')

    # Get Validation Data

    print ("getValidation", flush = True)
    golois.getValidation (input_data, policy, value, end)

    # Variable globale pour suivre la meilleure perte
    best_val_loss = float('inf')

    batches_per_epoch = N // batch

    clr = CyclicLR(
        base_lr=0.00005,
        max_lr=0.05,
        step_size=batches_per_epoch*2, # 2 epochs pour un cycle complet
        mode='triangular'
    )
    logger = LrLogger()

    optimizer = optimizers.legacy.SGD(momentum=0.9, nesterov=True, clipnorm=1.0)
    
    model.compile(
        optimizer=optimizer,
        loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
        loss_weights={'policy': policy_weight, 'value': value_weight},
        metrics={'policy': 'categorical_accuracy', 'value': 'mse'}
    )

    all_history = []
    val_loss_history = []
    

    for i in range(1, epochs + 1):
        #print(f'epoch {i}')

        golois.getBatch(input_data, policy, value, end, groups, i * N)

        history = model.fit(
            input_data,
            {'policy': policy, 'value': value},
            epochs=1,
            batch_size=batch,
            verbose=0,
            callbacks=[clr, logger]
        )

        metrics = {key: val[0] for key, val in history.history.items()}
        metrics['epoch'] = i
        all_history.append(metrics)
        # Affichage des métriques
        print(f"Epoch {i}: policy_loss={metrics['policy_loss']:.4f}, "
            f"value_loss={metrics['value_loss']:.4f}, "
            f"policy_categorical_accuracy={metrics['policy_categorical_accuracy']:.4f}, "
            f"value_mse={metrics['value_mse']:.4f}")
        
        if i % 5 == 0:
            gc.collect()
            # Évaluation du modèle sur les données de validation
            golois.getValidation(input_data, policy, value, end)
            val = model.evaluate(input_data, [policy, value], verbose=0, batch_size=batch)
            val_loss_history.append({
                'epoch': i,
                'val_policy_loss': val[1],
                'val_value_loss': val[2]
            })
            print(f"Validation: policy_loss={val[1]:.4f}, value_loss={val[2]:.4f}")
            
            current_val_loss = val[0]  # loss globale
            if current_val_loss < best_val_loss:
                print(f"Saving new best model at epoch {i} with val_loss={current_val_loss:.4f}")

                # Format propre du nom de fichier
                filename = f"best_model_epoch{i}_val{current_val_loss:.4f}.h5"
                model.save(filename)

                best_val_loss = current_val_loss

    total_time = time.time() - start_time
    return val, pd.DataFrame(all_history), pd.DataFrame(val_loss_history), total_time, logger.lrs

In [15]:
# Modèle 1 : GoMobileNetv2 (64,4,3) 
# - Nesterov  True 
# - SE True
# - Cyclyc lR
# - Batch size 32
# - Policy weight 1.0
# - Value weight 1.0

epochs=50

model = GoMobileNetv2((19,19,31), 64, 4, 3, True, activation=keras.activations.swish, drop_out_rate=0.3)

for layer in model.layers:
    if hasattr(layer, 'activation'):
        print(f"{layer.name}: activation = {layer.activation.__name__}")


conv2d_48: activation = linear
activation_72: activation = swish
conv2d_49: activation = linear
activation_73: activation = swish
depthwise_conv2d_18: activation = linear
conv2d_50: activation = linear
activation_74: activation = swish
dense_42: activation = swish
dense_43: activation = sigmoid
activation_75: activation = swish
conv2d_51: activation = linear
activation_76: activation = swish
depthwise_conv2d_19: activation = linear
conv2d_52: activation = linear
activation_77: activation = swish
dense_44: activation = swish
dense_45: activation = sigmoid
activation_78: activation = swish
conv2d_53: activation = linear
activation_79: activation = swish
depthwise_conv2d_20: activation = linear
conv2d_54: activation = linear
activation_80: activation = swish
dense_46: activation = swish
dense_47: activation = sigmoid
activation_81: activation = swish
conv2d_55: activation = linear
dense_48: activation = linear
activation_82: activation = swish
activation_83: activation = swish
policy: act

In [ ]:
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)
val, all_history, val_loss_history, total_time, lrs = train_model(
    model, 
    batch=32, 
    policy_weight=1.0, 
    value_weight=1.0, 
    epochs=epochs,
    N=10000
)

# Affichage des résultats
results = [
    (model, val, "Mon Model", total_time)
]
print_validation_results(results)

# Affichage de l'évolution du taux d'apprentissage
plot_learning_rate(lrs)

# Affichage des courbes comparatives
plot_result(
    history_dfs=[all_history], 
    val_dfs=[val_loss_history], 
    labels=["Mon Model"], 
    epochs=epochs
)